# EXP-26: Iterative Label Correction — Bold Cleaning

**Competition:** SPR 2026 — Mammography Report Classification  
**Base:** V12 pipeline (0.804) — 3 SVCs + 1 LGB  
**Enhancement:** Iterative label CORRECTION (not just removal)

**Approach (ousado):**
1. Rodar V12 com CV → OOF predictions
2. Amostras com P(label) < threshold E modelos concordam → CORRIGIR label (não remover)
3. Também: amostras com P(label) < threshold mais agressivo → REMOVER
4. Repetir 2-3 iterações (iterative refinement)
5. Retreinar V12 nos dados corrigidos
6. Threshold optimization + submission

**Vantagem sobre EXP-25:** Não perde amostras — corrige em vez de remover.
Iterativo: cada rodada limpa mais.
**Risco:** Se o modelo estiver errado, propaga erros.
**Runtime:** CPU only, ~15 min (múltiplas iterações)

In [ ]:
import os, re, hashlib, warnings, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import FeatureUnion
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, classification_report
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb
warnings.filterwarnings('ignore')

TRAIN_PATH = '/kaggle/input/competitions/spr-2026-mammography-report-classification/train.csv'
TEST_PATH  = '/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv'

train_orig = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

N_FOLDS = 5
N_CLASSES = 7
N_ITERATIONS = 3  # number of cleaning iterations

print(f'Train: {train_orig.shape}, Test: {test.shape}')
print(f'Class distribution:\n{train_orig["target"].value_counts().sort_index()}')

In [ ]:
def clean_achados(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r"\n{2,}", "\n", s)
    match = re.search(r'achados:(.*?)(análise comparativa:|$)', s, flags=re.DOTALL)
    if match: s = match.group(1).strip()
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def clean_full(s: str) -> str:
    if pd.isna(s): return ""
    s = str(s).strip().lower()
    s = s.replace("\r\n", "\n").replace("\r", "\n")
    s = re.sub(r"[ \t]+", " ", s)
    s = re.sub(r'[0-9]+,[0-9]+', 'NUM', s)
    s = re.sub(r'[0-9]+', 'NUM', s)
    return s

def stable_hash(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def extract_dense_features(df):
    features = pd.DataFrame(index=df.index)
    text_col = df['report'].fillna('').astype(str).str.lower()
    features['report_length']   = text_col.apply(len)
    features['has_measurement'] = text_col.str.contains(r'\b(?:cm|mm|medindo)\b', regex=True).astype(int)
    features['has_spiculation'] = text_col.str.contains(r'espiculad', regex=True).astype(int)
    features['has_distortion']  = text_col.str.contains(r'distorção arquitetural', regex=True).astype(int)
    features['has_biopsy']      = text_col.str.contains(r'biopsy|biópsia|resultado de cine|carcinoma', regex=True).astype(int)
    return csr_matrix(features.values)

def build_features(train_df, test_df):
    """Build all TF-IDF + dense features."""
    for df in [train_df, test_df]:
        df['achados'] = df['report'].apply(clean_achados)
        df['full']    = df['report'].apply(clean_full)
    
    tfidf_A = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=80000))
    ])
    X_tr_A = tfidf_A.fit_transform(train_df["achados"].values)
    X_te_A = tfidf_A.transform(test_df["achados"].values)
    
    tfidf_F = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=80000))
    ])
    X_tr_F = tfidf_F.fit_transform(train_df["full"].values)
    X_te_F = tfidf_F.transform(test_df["full"].values)
    
    tfidf_F2 = FeatureUnion([
        ("word", TfidfVectorizer(ngram_range=(1, 3), min_df=3, max_df=0.95, sublinear_tf=True)),
        ("char", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), min_df=3, max_df=0.95,
                                 sublinear_tf=True, max_features=100000))
    ])
    X_tr_F2 = tfidf_F2.fit_transform(train_df["full"].values)
    X_te_F2 = tfidf_F2.transform(test_df["full"].values)
    
    X_tr_dense = extract_dense_features(train_df)
    X_te_dense = extract_dense_features(test_df)
    X_tr_lgb = hstack([X_tr_A, X_tr_dense]).tocsr()
    X_te_lgb = hstack([X_te_A, X_te_dense]).tocsr()
    
    return {
        'A': (X_tr_A, X_te_A), 'F': (X_tr_F, X_te_F),
        'F2': (X_tr_F2, X_te_F2), 'lgb': (X_tr_lgb, X_te_lgb)
    }

def run_v12_cv(train_df, test_df, feats, verbose=True):
    """Run full V12 pipeline with CV, return OOF blend, test blend, per-model OOF."""
    y = train_df['target'].astype(int).values
    groups = train_df['report'].apply(stable_hash).values
    
    gkf = GroupKFold(n_splits=N_FOLDS)
    folds = list(gkf.split(feats['A'][0], y, groups))
    
    model_oof = {}
    model_test = {}
    
    for name, X_key in [('svc_A', 'A'), ('svc_F', 'F'), ('svc_F2', 'F2')]:
        X_tr, X_te = feats[X_key]
        oof = np.zeros((len(train_df), N_CLASSES), dtype=np.float64)
        te_acc = np.zeros((len(test_df), N_CLASSES), dtype=np.float64)
        for fi, (tr_idx, va_idx) in enumerate(folds):
            m = CalibratedClassifierCV(
                LinearSVC(class_weight="balanced", random_state=42, max_iter=10000),
                cv=3, method='sigmoid'
            )
            m.fit(X_tr[tr_idx], y[tr_idx])
            oof[va_idx] = m.predict_proba(X_tr[va_idx])
            te_acc += m.predict_proba(X_te) / N_FOLDS
        model_oof[name] = oof
        model_test[name] = te_acc
    
    X_lgb_tr, X_lgb_te = feats['lgb']
    oof = np.zeros((len(train_df), N_CLASSES), dtype=np.float64)
    te_acc = np.zeros((len(test_df), N_CLASSES), dtype=np.float64)
    for fi, (tr_idx, va_idx) in enumerate(folds):
        m = lgb.LGBMClassifier(class_weight='balanced', n_estimators=300, learning_rate=0.05,
                               max_depth=6, random_state=42, n_jobs=-1, verbose=-1)
        m.fit(X_lgb_tr[tr_idx], y[tr_idx])
        oof[va_idx] = m.predict_proba(X_lgb_tr[va_idx])
        te_acc += m.predict_proba(X_lgb_te) / N_FOLDS
    model_oof['lgb'] = oof
    model_test['lgb'] = te_acc
    
    oof_blend = 0.70 * (0.25 * model_oof['svc_A'] + 0.40 * model_oof['svc_F'] + 0.35 * model_oof['svc_F2']) + 0.30 * model_oof['lgb']
    te_blend = 0.70 * (0.25 * model_test['svc_A'] + 0.40 * model_test['svc_F'] + 0.35 * model_test['svc_F2']) + 0.30 * model_test['lgb']
    
    oof_f1 = f1_score(y, np.argmax(oof_blend, axis=1), average='macro')
    if verbose:
        print(f"  V12 OOF F1: {oof_f1:.4f}")
    
    return oof_blend, te_blend, model_oof, y

print("Functions defined.")

In [ ]:
# =============================================================================
# Iterative Label Correction Loop
# =============================================================================

# Thresholds for label actions:
CORRECT_THRESHOLD = 0.05   # P(label) < 5% AND all models agree → CORRECT the label
REMOVE_THRESHOLD = 0.03    # P(label) < 3% AND models disagree on alt → REMOVE (too confusing)

train_current = train_orig.copy()
total_corrected = 0
total_removed = 0

for iteration in range(N_ITERATIONS):
    print(f"\n{'='*60}")
    print(f"ITERATION {iteration + 1}/{N_ITERATIONS}")
    print(f"{'='*60}")
    print(f"Current train size: {len(train_current)}")
    
    train_current = train_current.reset_index(drop=True)
    feats = build_features(train_current, test)
    oof_blend, te_blend, model_oof, y_current = run_v12_cv(train_current, test, feats)
    
    # Analyze labels
    oof_label_prob = oof_blend[np.arange(len(y_current)), y_current]
    oof_pred = np.argmax(oof_blend, axis=1)
    
    # Per-model predictions
    model_preds = {name: np.argmax(oof, axis=1) for name, oof in model_oof.items()}
    alt_classes = np.stack(list(model_preds.values()), axis=1)
    all_models_disagree = np.all(alt_classes != y_current[:, None], axis=1)
    all_models_agree_alt = np.all(alt_classes == alt_classes[:, 0:1], axis=1)
    
    # CORRECT: P(label) < CORRECT_THRESHOLD, all models disagree, all agree on same alt
    correct_mask = (oof_label_prob < CORRECT_THRESHOLD) & all_models_disagree & all_models_agree_alt
    n_correct = correct_mask.sum()
    
    # REMOVE: P(label) < REMOVE_THRESHOLD, all models disagree, but DON'T agree on alt
    remove_mask = (oof_label_prob < REMOVE_THRESHOLD) & all_models_disagree & ~all_models_agree_alt
    n_remove = remove_mask.sum()
    
    print(f"\n  Samples to CORRECT: {n_correct}")
    print(f"  Samples to REMOVE:  {n_remove}")
    
    if n_correct == 0 and n_remove == 0:
        print("  No more changes needed. Stopping.")
        break
    
    # Apply corrections
    if n_correct > 0:
        new_labels = oof_pred[correct_mask]
        old_labels = y_current[correct_mask]
        print(f"\n  Corrections (old → new):")
        for old, new in zip(old_labels[:15], new_labels[:15]):
            print(f"    class {old} → class {new}")
        train_current.loc[correct_mask, 'target'] = new_labels
        total_corrected += n_correct
    
    # Apply removals
    if n_remove > 0:
        train_current = train_current[~remove_mask]
        total_removed += n_remove
    
    print(f"\n  Running totals: {total_corrected} corrected, {total_removed} removed")
    print(f"  New class distribution:\n{train_current['target'].value_counts().sort_index()}")

print(f"\n{'='*60}")
print(f"CLEANING COMPLETE: {total_corrected} corrected, {total_removed} removed")
print(f"Final train size: {len(train_current)} (original: {len(train_orig)})")

In [ ]:
# =============================================================================
# Final training on corrected data
# =============================================================================
print("Final training on corrected data...")

train_final = train_current.reset_index(drop=True)
feats_final = build_features(train_final, test)
oof_final, te_final, _, y_final = run_v12_cv(train_final, test, feats_final)

baseline_preds = np.argmax(oof_final, axis=1)
baseline_f1 = f1_score(y_final, baseline_preds, average='macro')
print(f"\nFinal OOF F1 (no thresholds): {baseline_f1:.4f}")
print(classification_report(y_final, baseline_preds, digits=4))

In [ ]:
# =============================================================================
# Threshold optimization
# =============================================================================

threshold_classes = [6, 5, 4, 3]
threshold_range = np.arange(0.05, 0.55, 0.01)

def apply_thresholds(proba, thresholds):
    preds = np.argmax(proba, axis=1).copy()
    for cls in [6, 5, 4, 3]:
        if cls in thresholds:
            mask = proba[:, cls] > thresholds[cls]
            for higher_cls in [c for c in [6, 5, 4, 3] if c != cls]:
                if higher_cls in thresholds and threshold_classes.index(higher_cls) < threshold_classes.index(cls):
                    mask = mask & (preds != higher_cls)
            preds[mask] = cls
    return preds

best_thresholds = {}
current_f1 = baseline_f1

for cls in threshold_classes:
    best_cls_f1 = current_f1
    best_cls_thr = None
    for thr in threshold_range:
        trial = {**best_thresholds, cls: thr}
        trial_preds = apply_thresholds(oof_final, trial)
        trial_f1 = f1_score(y_final, trial_preds, average='macro')
        if trial_f1 > best_cls_f1:
            best_cls_f1 = trial_f1
            best_cls_thr = thr
    if best_cls_thr is not None:
        best_thresholds[cls] = best_cls_thr
        current_f1 = best_cls_f1
        print(f"Class {cls}: threshold={best_cls_thr:.2f}, macro-F1={best_cls_f1:.4f}")
    else:
        print(f"Class {cls}: no improvement")

final_f1 = f1_score(y_final, apply_thresholds(oof_final, best_thresholds), average='macro')
print(f"\nFinal OOF F1 with thresholds: {final_f1:.4f}")
print(f"Thresholds: {best_thresholds}")

In [ ]:
# =============================================================================
# Submission
# =============================================================================

test_preds = apply_thresholds(te_final, best_thresholds)

def apply_safe_rules(row):
    text = str(row['report']).lower()
    current_pred = int(row['target'])
    if re.search(r'resultado de cine grau 3|carcinoma|\bcdis\b', text):
        return 6
    if 'espiculad' in text and 'distorção' in text and current_pred < 4:
        return 5
    return current_pred

test['target'] = test_preds
test['target'] = test.apply(apply_safe_rules, axis=1)

submission = test[['ID', 'target']].copy()
submission['target'] = submission['target'].astype(int)
submission.to_csv('submission.csv', index=False)

print('Prediction distribution:')
print(submission['target'].value_counts().sort_index())
print(f'\nSubmission saved! Shape: {submission.shape}')
print(submission)